[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1rLxam_00xSpOwsmW7_e3oZ12hwy_y7Hk/view?usp=drive_link)

# Prompt Evaluation – With RAG Contexts

This notebook demonstrates how to compare prompts when your pipeline uses retrieved context. Samples include `contexts`; Floeval generates responses from question + context for each prompt, then scores with faithfulness.

**Objectives**
- Install Floeval and configure credentials
- Create a prompts file and dataset with `contexts`
- Run prompt evaluation with RAG metrics (answer_relevancy, faithfulness)
- Inspect results by prompt

## 1. Create the Prompts File

A YAML file is created with prompt variants that instruct the model how to use the retrieved context.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
from pathlib import Path

prompts_content = """
prompts:
  "1":
    template: "Answer using only the provided context. Cite which parts you use."
  "2":
    template: "Summarize the provided context to answer the question."
  "3":
    template: "Answer the question based on the context. Be concise and accurate."
"""
Path("prompts_rag.yaml").write_text(prompts_content.strip())
print("Created prompts_rag.yaml")

## 2. Imports

The following cell imports the evaluation components and the LLM configuration schema.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 3. Configure the LLM

The LLM configuration is built using environment variables. Set `OPENAI_API_KEY` in your environment or replace the placeholder.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 4. Load the Dataset with Contexts

The dataset is loaded with `user_input`, `contexts` (retrieved documents), and `prompt_ids` for each sample. Floeval generates responses from question and context for each prompt.

In [ ]:
dataset = DatasetLoader.from_samples(
    [
        {
            "user_input": "What is RAG?",
            "contexts": ["RAG combines document retrieval with language generation."],
            "prompt_ids": ["1", "2", "3"],
        },
    ],
    partial_dataset=True,
)
print(f"Dataset loaded: {len(dataset.samples)} samples")

## 5. Create and Run the Evaluation

The `answer_relevancy` and `faithfulness` metrics are used to compare how well each prompt produces grounded answers.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy", "faithfulness"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file="prompts_rag.yaml",
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

## 6. Inspect Results by Prompt

Each result includes `prompt_id` and the generated `llm_response`. Results are inspected by prompt to compare faithfulness across instructions.

In [ ]:
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    metrics = sr.get("metrics", {})
    print(f"Prompt: {pid}")
    for k, v in metrics.items():
        print(f"  {k}: {v.get('score')}")

## Summary

This notebook demonstrated how to compare prompts when the pipeline uses retrieved context.

The key components included:

1. **Prompts File**: A YAML file with context-aware instructions was created.
2. **Dataset with Contexts**: A partial dataset was loaded with `user_input`, `contexts`, and `prompt_ids`.
3. **RAG Metrics**: The `answer_relevancy` and `faithfulness` metrics were used to evaluate grounding quality.
4. **Results by Prompt**: Results were inspected by `prompt_id` to compare grounding quality across instructions.

This example showcases prompt evaluation with RAG contexts and faithfulness scoring.